# Etapa 20 — Respuestas catalográficas verificables y revisión por afirmación

**Objetivo:** pasar de citas formalmente válidas a una presentación más cauta. Este notebook distingue:

1. **El campo del catálogo fue comprobado:** el fragmento existe literalmente en la ficha recuperada.
2. **La afirmación está respaldada:** requiere revisión humana y, en ocasiones, lectura del libro o documento original.
3. **Respuesta catalográfica conservadora:** enumera únicamente títulos, unidades temáticas y enlaces conocidos; **no** deduce el contenido de las obras.

No utiliza Groq, no ejecuta OCR y no vuelve a descargar fuentes. Lee `etapa19.zip` automáticamente. Ejecuta las celdas en orden y se creará `data/output/etapa20.zip`.

> La etiqueta `cita_y_evidencia_literal` no equivale a «afirmación históricamente cierta». Ninguna revisión humana se inventa ni se da por completada.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from io import BytesIO
from collections import Counter
import json, re, zipfile, hashlib, unicodedata
import pandas as pd

HERE=Path.cwd().resolve()
ROOT=next((p for p in (HERE,*HERE.parents) if (p/'notebooks').is_dir() and (p/'data').is_dir()),None)
if ROOT is None:
    raise FileNotFoundError('Abre este notebook en un proyecto con notebooks/ y data/.')
ENTRADA=next((p for p in (ROOT/'data/input/etapa19.zip',ROOT/'data/output/etapa19.zip') if p.is_file()),None)
if ENTRADA is None:
    raise FileNotFoundError('No encuentro etapa19.zip en data/input ni data/output.')
DEST=ROOT/'data/output/etapa20'
DEST.mkdir(parents=True,exist_ok=True)
NECESARIOS={'auditoria_afirmaciones.csv','fuentes_recuperadas.csv','revision_historica.csv','informe_etapa19.json'}
with zipfile.ZipFile(ENTRADA) as z:
    faltan=NECESARIOS-set(z.namelist())
    if faltan: raise ValueError('Faltan archivos en etapa19.zip: '+str(faltan))
    def leer_csv(nombre):
        return pd.read_csv(BytesIO(z.read(nombre)),dtype=str,keep_default_na=False,encoding='utf-8-sig')
    afirmaciones=leer_csv('auditoria_afirmaciones.csv')
    fuentes=leer_csv('fuentes_recuperadas.csv')
    revision_inicial=leer_csv('revision_historica.csv')
    resumen19=json.loads(z.read('informe_etapa19.json').decode('utf8'))

assert len(afirmaciones)==len(afirmaciones.afirmacion_id.unique()),'IDs de afirmación repetidos'
assert set(afirmaciones.pregunta).issubset(set(fuentes.pregunta)),'Hay preguntas sin contexto'
print('Proyecto:',ROOT)
print('Entrada:',ENTRADA.name)
print('Afirmaciones:',len(afirmaciones),'| Preguntas:',afirmaciones.pregunta.nunique(),'| Fichas recuperadas:',fuentes.mcid.nunique())

Proyecto: C:\Users\jvera\Documents\memoria_chilena
Entrada: etapa19.zip
Afirmaciones: 16 | Preguntas: 5 | Fichas recuperadas: 17


## 1. Auditoría independiente de citas y fragmentos

Para cada afirmación comprobamos que los IDs citados estuvieran **en el contexto de su pregunta** y que cada fragmento de evidencia exista en el campo exacto del registro recuperado. Las menciones sobre contenidos, fechas, exclusividad o géneros no se aprueban automáticamente.

In [2]:
CAMPOS={'titulo','autor','fecha_catalografica','materias','unidades_tematicas','notas','coleccion','derechos'}

def normalizar(t):
    return re.sub(r'\s+',' ',unicodedata.normalize('NFC',str(t or ''))).strip().casefold()

def separar_ids(citas):
    return sorted({x.strip() for x in str(citas).split('|') if re.fullmatch(r'MC\d{5,}',x.strip())})

# Contexto indexado POR PREGUNTA; no compartir una evidencia con otras consultas.
indice={(r.pregunta,r.mcid):r for r in fuentes.itertuples(index=False)}

def evidencia_literal_valida(pregunta,mcid,evidencia):
    ficha=indice.get((pregunta,mcid))
    if ficha is None or not isinstance(evidencia,dict): return False
    campo=str(evidencia.get('campo',''))
    frag=str(evidencia.get('fragmento',''))
    if campo not in CAMPOS or not frag.strip():return False
    if str(evidencia.get('mcid',''))!=mcid:return False
    if normalizar(frag) not in normalizar(getattr(ficha,campo,'')):return False
    enlace=str(evidencia.get('ficha',''))
    if enlace and enlace != str(ficha.ficha):return False
    return True

riesgos_especificos={
 'Q01-A01':'La ley electoral aparece por asociación temática: no afirmar que su contenido regule directamente el voto femenino.',
 'Q02-A02':'La fecha catalográfica 1953 no acredita que la obra fuera catalogada ese año.',
 'Q02-A04':'No concluir que solo un documento de la colección estudia ese tema.',
 'Q03-A03':'Actas del Cabildo de 1810 no demuestra relación con celebraciones de 1910.',
 'Q04-A01':'El título de Doctrina christiana no basta para demostrar qué lenguas aparecen en sus páginas.',
 'Q05-A01':'Compartir cronistasperuanos no prueba el género literario de Extirpación de la idolatría.',
 'Q05-A02':'La referencia MC0013341 no tiene evidencia literal aportada para esa afirmación.'
}

filas=[]
for r in afirmaciones.itertuples(index=False):
    citas=separar_ids(r.citas)
    try: evidencias=json.loads(r.evidencias_json or '[]')
    except json.JSONDecodeError: evidencias=[]
    validos=[]
    for mcid in citas:
        validos.extend([e for e in evidencias if evidencia_literal_valida(r.pregunta,mcid,e)])
    ids_validos={e['mcid'] for e in validos}
    fuera_contexto=[mcid for mcid in citas if (r.pregunta,mcid) not in indice]
    sin_fragmento=[mcid for mcid in citas if mcid not in ids_validos]
    completo=bool(citas) and not fuera_contexto and not sin_fragmento
    # Un 'sí' aquí solo significa que comprobamos literales, NO toda la oración.
    if completo:
        estado='CITA_Y_CAMPO_VERIFICADOS_NO_VERDAD_HISTORICA'
    elif fuera_contexto:
        estado='CITA_FUERA_DEL_CONTEXTO'
    elif sin_fragmento:
        estado='CITA_SIN_FRAGMENTO_COMPROBADO'
    else:
        estado='AFIRMACION_SIN_CITA_PROPIA'
    alertas=[]
    if r.alertas_inferencia:alertas.append(r.alertas_inferencia)
    if r.afirmacion_id in riesgos_especificos:alertas.append(riesgos_especificos[r.afirmacion_id])
    filas.append({
        'afirmacion_id':r.afirmacion_id,'pregunta_id':r.pregunta_id,'pregunta':r.pregunta,
        'afirmacion':r.afirmacion,'citas':r.citas,
        'estado_comprobacion':estado,'n_citas':len(citas),
        'n_citas_con_fragmento_comprobado':len(ids_validos),
        'citas_fuera_contexto':' | '.join(fuera_contexto),
        'citas_sin_fragmento':' | '.join(sin_fragmento),
        'alertas':' | '.join(alertas),
        'conclusion':'REQUIERE_REVISION_HISTORICA',
        'evidencias_verificadas_json':json.dumps(validos,ensure_ascii=False),
    })
control=pd.DataFrame(filas)
print(control.estado_comprobacion.value_counts().to_string())
assert len(control)==len(afirmaciones)
assert not any(control.conclusion!='REQUIERE_REVISION_HISTORICA')
assert control.loc[control.afirmacion_id=='Q05-A02','estado_comprobacion'].iloc[0]=='CITA_SIN_FRAGMENTO_COMPROBADO'

estado_comprobacion
AFIRMACION_SIN_CITA_PROPIA                      8
CITA_Y_CAMPO_VERIFICADOS_NO_VERDAD_HISTORICA    7
CITA_SIN_FRAGMENTO_COMPROBADO                   1


## 2. Respuestas prudentes basadas en fichas

Estas **no sustituyen la investigación histórica**. Sirven como respuestas de descubrimiento documental: solo muestran datos comprobables (título, ficha, colección y relaciones curatoriales). Se excluyen, por precaución, las asociaciones poco pertinentes que detectamos en la auditoría.

In [3]:
SELECCION={
  'Q01': [('MC0061013','Título directamente relacionado con el voto femenino'),
          ('MC0031311','Título directamente relacionado con el sufragio femenino'),
          ('MC0077873','Asociación catalográfica con la colección; no presupone el contenido de la ley')],
  'Q02': [('MC0000958','Título expresamente dedicado a la fundación del Instituto Nacional')],
  'Q03': [('MC0001859','Título sobre los festejos del centenario'),
          ('MC0049519','Publicación periódica fechada en 1910 según el título'),
          ('MC0027682','Título sobre Chile en 1910; no presupone que describa los festejos')],
  'Q04': [('MC0014761','Título directamente relacionado con la lengua aymara'),
          ('MC0014753','Asociación curatorial con el pueblo aymara; verificar las lenguas en la obra')],
  'Q05': [('MC0014754','Título de crónica; asociación catalográfica cronistasperuanos'),
          ('MC0014755','Título de Comentarios reales; asociación catalográfica cronistasperuanos'),
          ('MC0014760','Solo asociación curatorial cronistasperuanos; no prueba género crónica')],
}

preguntas_por_id=afirmaciones[['pregunta_id','pregunta']].drop_duplicates().set_index('pregunta_id')['pregunta'].to_dict()
respuestas_prudentes=[]
secciones=['# Respuestas de descubrimiento documental (metadatos verificados)',
           '> Texto producido mediante plantillas deterministas. No se han leído ni comprobado los contenidos completos de los libros.']
for qid,pairs in SELECCION.items():
    pregunta=preguntas_por_id[qid]
    secciones += ['', '## '+pregunta, '', 'El catálogo ofrece las siguientes fichas pertinentes:']
    for mcid,razon in pairs:
        registro=indice.get((pregunta,mcid))
        if registro is None:
            raise ValueError(f'La ficha {mcid} no estaba en el contexto de {qid}; no se puede citar.')
        titulo=str(registro.titulo).strip()
        link=str(registro.ficha).strip()
        if not link.startswith('https://www.memoriachilena.gob.cl/'):
            raise ValueError(f'Enlace inesperado para {mcid}: {link}')
        secciones.append(f'- **[{titulo}]({link})** [{mcid}]. {razon}.')
        respuestas_prudentes.append({'pregunta_id':qid,'pregunta':pregunta,
            'mcid':mcid,'titulo':titulo,'ficha':link,'coleccion':registro.coleccion,
            'justificacion_catalografica':razon,'nivel_evidencia':'METADATOS_UNICAMENTE'})
    secciones += ['', 'Nota: las relaciones temáticas proceden del catálogo; no demuestran por sí solas el contenido o las conclusiones de cada obra.']
prudentes=pd.DataFrame(respuestas_prudentes)
texto_prudente='\n'.join(secciones)+'\n'
print('Respuestas para',len(SELECCION),'preguntas; fichas citadas:',len(prudentes))

Respuestas para 5 preguntas; fichas citadas: 12


## 3. Revisión humana (sin widgets)

El notebook **no marca ninguna afirmación como históricamente aprobada**. Si deseas registrar revisiones desde Jupyter, usa la función `anotar_revision('Q01-A01', 1, '...')` y después `guardar_etapa20()`.

Escala: `0` = no sustentada, `1` = sustento parcial o necesita fuente primaria, `2` = respaldada tras revisión de la fuente. Se permite dejar la evaluación en blanco. **No uses 2 solo porque el título coincide.**

In [4]:
RUTA_REVISION=DEST/'revision_historica.csv'
columnas_revision=['afirmacion_id','pregunta','afirmacion','citas','estado_comprobacion','juicio_humano','notas_revision','fuente_primaria_verificada']
plantilla=control[['afirmacion_id','pregunta','afirmacion','citas','estado_comprobacion']].copy()
plantilla['juicio_humano']=''
plantilla['notas_revision']=''
plantilla['fuente_primaria_verificada']=''

# Preserva las revisiones escritas en el CSV durante ejecuciones anteriores.
if RUTA_REVISION.is_file():
    anterior=pd.read_csv(RUTA_REVISION,dtype=str,keep_default_na=False)
else:
    anterior=revision_inicial.copy()
if 'afirmacion_id' in anterior.columns:
    previo=anterior.drop_duplicates('afirmacion_id').set_index('afirmacion_id')
    for col in ['juicio_humano','notas_revision','fuente_primaria_verificada']:
        if col in previo.columns:
            plantilla[col]=plantilla.afirmacion_id.map(previo[col]).fillna('').astype(str)
REVISION=plantilla[columnas_revision].copy()

def anotar_revision(afirmacion_id, juicio, nota, fuente_primaria=''):
    """Guarda una anotación explícita del investigador; nunca se autocalifica."""
    if afirmacion_id not in set(REVISION.afirmacion_id):
        raise ValueError('ID desconocido: '+str(afirmacion_id))
    if str(juicio) not in ('0','1','2'):
        raise ValueError('Usa 0, 1 o 2. Para dejar pendiente, no anotes.')
    mask=REVISION.afirmacion_id==afirmacion_id
    REVISION.loc[mask,'juicio_humano']=str(juicio)
    REVISION.loc[mask,'notas_revision']=str(nota)
    REVISION.loc[mask,'fuente_primaria_verificada']=str(fuente_primaria)
    REVISION.to_csv(RUTA_REVISION,index=False,encoding='utf-8-sig')
    print('Guardada revisión de',afirmacion_id)

print('Revisiones humanas existentes:',sum(REVISION.juicio_humano.isin(['0','1','2'])),'de',len(REVISION))
print('Ejemplo (no se ejecuta): anotar_revision("Q01-A01", 1, "Verificar el alcance de la ley en el texto original")')

Revisiones humanas existentes: 0 de 16
Ejemplo (no se ejecuta): anotar_revision("Q01-A01", 1, "Verificar el alcance de la ley en el texto original")


## 4. Exportación automática

El ZIP contiene la matriz de control, una plantilla de revisión, respuestas documentales prudentes y un informe metodológico. No incluye PDF, claves de API ni nuevos resultados de un LLM.

In [5]:
def guardar_etapa20():
    control.to_csv(DEST/'matriz_control_afirmaciones.csv',index=False,encoding='utf-8-sig')
    prudentes.to_csv(DEST/'fichas_respuestas_prudentes.csv',index=False,encoding='utf-8-sig')
    (DEST/'respuestas_catalograficas_prudentes.md').write_text(texto_prudente,encoding='utf-8')
    REVISION.to_csv(RUTA_REVISION,index=False,encoding='utf-8-sig')
    evaluadas=int(REVISION.juicio_humano.isin(['0','1','2']).sum())
    d={
     'etapa':20,'fuente':'etapa19.zip',
     'fecha_utc':datetime.now(timezone.utc).isoformat(),
     'preguntas':len(SELECCION),
     'afirmaciones':len(control),
     'citas_y_campos_verificados':int((control.estado_comprobacion=='CITA_Y_CAMPO_VERIFICADOS_NO_VERDAD_HISTORICA').sum()),
     'afirmaciones_sin_comprobacion_literal_completa':int((control.estado_comprobacion!='CITA_Y_CAMPO_VERIFICADOS_NO_VERDAD_HISTORICA').sum()),
     'revision_humana_completada':evaluadas,
     'afirmaciones_sin_revision_humana':int(len(REVISION)-evaluadas),
     'respuestas_catalograficas':len(SELECCION),
     'fichas_en_respuestas':len(prudentes),
     'llamadas_groq':0,
     'llamadas_ocr':0,
     'limite':'La coincidencia literal de metadatos no acredita la veracidad histórica de la afirmación; las respuestas prudentes sirven para descubrimiento documental, no para validar el contenido de libros.'
    }
    (DEST/'informe_etapa20.json').write_text(json.dumps(d,ensure_ascii=False,indent=2),encoding='utf-8')
    info=f"""# Informe de etapa 20

- Afirmaciones de etapa 19: **{d['afirmaciones']}**.
- Con citas y campos verificables: **{d['citas_y_campos_verificados']}**.
- Sin comprobación literal completa: **{d['afirmaciones_sin_comprobacion_literal_completa']}**.
- Revisiones humanas registradas: **{evaluadas}/{len(REVISION)}**.
- Respuestas catalográficas prudentes: **{len(SELECCION)}** con **{len(prudentes)}** referencias a fichas.

**Observación:** una cita literal no valida toda la afirmación. La ficha MC0013341 queda fuera de la respuesta a crónicas peruanas; su presencia en el contexto recuperado no justifica una asociación histórica. Tampoco se interpreta una fecha de catálogo como fecha de catalogación y se evita afirmar que la Ley de Elecciones de 1934 regula directamente el voto femenino sin comprobar su contenido.

Esta etapa no genera afirmaciones históricas nuevas con un LLM ni aplica OCR. Para aprobar afirmaciones por contenido, revisar fuentes primarias y registrar la fuente en `revision_historica.csv`.
"""
    (DEST/'INFORME_ETAPA20.md').write_text(info,encoding='utf-8')
    zipdest=ROOT/'data/output/etapa20.zip'
    nombres=sorted(p for p in DEST.iterdir() if p.is_file())
    with zipfile.ZipFile(zipdest,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in nombres:z.write(p,p.name)
    print('ZIP generado:',zipdest)
    print('Archivos:',', '.join(p.name for p in nombres))
    return zipdest

ZIP_FINAL=guardar_etapa20()

ZIP generado: C:\Users\jvera\Documents\memoria_chilena\data\output\etapa20.zip
Archivos: fichas_respuestas_prudentes.csv, informe_etapa20.json, INFORME_ETAPA20.md, matriz_control_afirmaciones.csv, respuestas_catalograficas_prudentes.md, revision_historica.csv
